# `avh` — AV-HuBERT features into the shared cache

Turns every clip into two fixed-length token sequences from **AV-HuBERT** (Meta, 2022), the
audio-visual speech encoder trained on LRS3 + VoxCeleb2:

```
<DATA>/cached_features/avhubert-video/<clip_id>.npy   float16  (64, 768)
<DATA>/cached_features/avhubert-audio/<clip_id>.npy   float16  (64, 768)
```

Both streams come from the **same checkpoint**: the mouth at 25 fps and the log-mel filterbank
of the same speech, each through the encoder in its own modality. AV-HuBERT was trained with
modality dropout, so a mouth-only and an audio-only pass are both in-distribution.

> **Why this notebook builds a second Python.** The checkpoint is a fairseq model, and fairseq
> 0.12.2 is the last release that can load it — its wheels stop at **cp38**, while Colab runs
> 3.13. So the encoder lives in a Python 3.8 environment at `/content/avh8`, created once by
> micromamba, and the extraction runs there as a subprocess. This kernel never imports fairseq.

> **And the licence.** The weights are under Meta's AV-HuBERT licence: non-commercial research,
> explicitly no biometric processing or surveillance use — read [`docs/plan.md`](docs/plan.md).

| | |
|---|---|
| **Time** | ~5 min to build the 3.8 environment, then ~2–3 h of T4 for the protocol |
| **Writes** | `cached_features/avhubert-{video,audio}/` + one `manifest.json` each |
| **Reads** | `clips/`, `audio/`, `dataset.csv`, `splits/` |
| **Resumable** | yes — an existing `.npy` is skipped |

## 0 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
if not os.path.isdir("/content/DeepLearningCourse/.git"):
    !git clone -q https://github.com/mattsays/DeepLearningCourse /content/DeepLearningCourse
%cd /content/DeepLearningCourse/lie-detector
!git pull -q
# AV-HuBERT carries its own fairseq fork as a submodule; the 3.8 environment below is built
# from that fork only if the wheel path ever stops working.
!git submodule update --init --recursive --depth 1 vendor/av_hubert
import sys; sys.path.insert(0, '.')
print(sorted(os.listdir('vendor/av_hubert'))[:8])

## 1 — The Python 3.8 environment

Colab's kernel is 3.13 and fairseq stops at cp38. `micromamba` gives a self-contained 3.8 at
`/content/avh8` without touching the kernel, and `fairseq==0.12.2` installs there **from its
cp38 wheel** — no Cython build, which is the thing that fails on newer Pythons.

`dlib` comes from conda-forge because `dlib-bin` has no cp38 wheel. Everything else is pip.

The cell is idempotent: if `/content/avh8` already exists it only re-checks the imports, so
re-running it after a disconnect costs seconds.

In [ ]:
%%bash
set -e
MM=/content/micromamba/bin/micromamba
if [ ! -x "$MM" ]; then
  mkdir -p /content/micromamba
  curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest \
    | tar -xj -C /content/micromamba --strip-components=1 bin/micromamba
fi

if [ ! -x /content/avh8/bin/python ]; then
  "$MM" create -y -p /content/avh8 python=3.8 -c conda-forge
  "$MM" install -y -p /content/avh8 -c conda-forge dlib
fi

PY=/content/avh8/bin/python
$PY -m pip install -q --upgrade pip
# torch 2.4.1 is the last release with cp38 wheels; the cu121 build works on a T4.
$PY -m pip install -q torch==2.4.1 torchaudio==2.4.1 \
  --index-url https://download.pytorch.org/whl/cu121
$PY -m pip install -q "numpy<2" opencv-python-headless soundfile \
  python_speech_features scikit-learn
$PY -m pip install -q --no-deps fairseq==0.12.2
$PY -m pip install -q "hydra-core>=1.0.7,<1.1" "omegaconf<2.1" bitarray sacremoses regex tqdm

$PY - <<'CHECK'
import fairseq, torch, cv2, dlib, numpy
print("python 3.8 env ready")
print(" fairseq", fairseq.__version__, "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available(), "| numpy", numpy.__version__)
CHECK

## 2 — What gets encoded, and how

`scripts/extract_avhubert.py` owns the pass: it reads the frozen splits, the paths from
`dataset.csv`, and writes both caches. The preprocessing itself lives in
[`preprocess.py`](preprocess.py) — the same file this notebook imports for the crop check, so
the frames you look at are the frames that were cached.

The three steps, for the record:

1. decode the clip and sample to **25 fps**;
2. detect the face with dlib's HOG detector once per second, predict 68 landmarks on every
   frame, take mouth points 48–67, expand the box, crop to **96×96 grayscale**;
3. for the audio pass, `logfbank` (26 bins, 10 ms hop), four frames stacked to **104 dims**, a
   per-frame layer norm — exactly `hubert_dataset.py`.

## 3 — Look at the crops before spending two hours

`--self-test` runs the preprocessing on one clip and writes a PNG grid: no encoder, no cache.
If the mouth is not centred here, stop — the encoder would be reading the chin.

In [ ]:
import sys
from pathlib import Path

DATA = Path("/content/drive/MyDrive/dolos")
CHECKPOINT = Path("/content/drive/MyDrive/dolos/checkpoints/base_vox_iter5.pt")
LANDMARKS = Path("/content/drive/MyDrive/dolos/checkpoints/shape_predictor_68_face_landmarks.dat")
GRID = Path("/content/avh_grid.png")

print("checkpoint", CHECKPOINT, "exists" if CHECKPOINT.exists() else "MISSING")
print("landmarks ", LANDMARKS, "exists" if LANDMARKS.exists() else "MISSING")

In [ ]:
!cd /content/DeepLearningCourse/lie-detector && /content/avh8/bin/python scripts/extract_avhubert.py   --data "{DATA}"   --checkpoint "{CHECKPOINT}"   --landmarks "{LANDMARKS}"   --self-test --dump-grid /content/avh_grid.png

In [ ]:
from PIL import Image
Image.open("/content/avh_grid.png")

## 4 — Smoke test: three clips

Downloads nothing, caches three clips in both folders, so the whole path — decode, crop,
filterbank, two forward passes, manifest — is exercised before committing to a session.

In [ ]:
!cd /content/DeepLearningCourse/lie-detector && /content/avh8/bin/python scripts/extract_avhubert.py   --data "{DATA}"   --checkpoint "{CHECKPOINT}"   --landmarks "{LANDMARKS}"   --limit 3

## 5 — Full run

~1,675 clips, two passes each. A disconnect is not a disaster: rerun the cell and it continues
from the last written array. `--force` rebuilds, and is refused silently by the manifest if the
settings changed — the script then tells you to use it.

In [ ]:
!cd /content/DeepLearningCourse/lie-detector && /content/avh8/bin/python scripts/extract_avhubert.py   --data "{DATA}"   --checkpoint "{CHECKPOINT}"   --landmarks "{LANDMARKS}"

## 6 — Verify

Every split clip has both arrays, shapes are `(64, 768)`, and a logistic probe on one fold tells
a working cache from a broken one. The probe is a plumbing check, not a result.

In [ ]:
import json
import numpy as np
from pathlib import Path

from lib.dataset import Clips, Protocol, cache_dir

DATA = Path("/content/drive/MyDrive/dolos")
folds = Protocol.load(DATA / "splits")
clips = Clips.load(DATA)

for name in ("avhubert-video", "avhubert-audio"):
    directory = cache_dir(name, DATA)
    files = sorted(directory.glob("*.npy"))
    size_mb = sum(path.stat().st_size for path in files) / 1e6
    manifest = json.loads((directory / "manifest.json").read_text())
    sample = np.load(files[0]) if files else np.zeros((0,))
    print(f"{name:16} {len(files):5d} files  {size_mb:7.1f} MB  shape {sample.shape}  "
          f"failures {len(manifest.get('failures', {}))}")

directory = cache_dir("avhubert-audio", DATA)
train_ids, val_ids, _ = folds.subject(1)
train_ids = [c for c in train_ids if (directory / f"{c}.npy").exists()]
val_ids = [c for c in val_ids if (directory / f"{c}.npy").exists()]

if train_ids and val_ids:
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    x_train = np.stack([np.load(directory / f"{c}.npy").mean(0) for c in train_ids])
    x_val = np.stack([np.load(directory / f"{c}.npy").mean(0) for c in val_ids])
    probe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=0.1))
    probe.fit(x_train, clips.labels(train_ids))
    print(f"audio-only probe on fold 1 val: "
          f"AUC {roc_auc_score(clips.labels(val_ids), probe.predict_proba(x_val)[:, 1]):.3f}")
print("(plumbing check — the number that counts comes from the five folds in train.ipynb)")

## 7 — Next

`train.ipynb` reads `avhubert-video` + `avhubert-audio`. Both streams are 768 wide and come
from the same encoder, so `unified` fuses them outside the model and `congruence` compares two
representations that were trained to agree.

The model's own fused pass (`modalities = ["audio", "video"]`, one forward for both) is not
cached: it emits a single sequence, which would make a two-stream probe impossible. If you want
that baseline, add a third cache and a `fused` variant.